<a href="https://colab.research.google.com/github/JayJ17/Heart-Academy-Project-1/blob/main/heart_of_code_unit_2_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#google/gemma-3-4b-it
#google/gemma-4-31b-it
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from google.colab import userdata
import os
from openai import OpenAI
from getpass import getpass

#For Cosine Sim
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. SETUP: Get API Key and Initialize Client
API_KEY = userdata.get('ApiKey')

# Initialize the OpenAI client correctly for OpenRouter
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=API_KEY,
)






# ==========================================
# 2. SYSTEM PROMPT: The App's "Brain"
# ==========================================
APP_PERSONA = """
You are a math and physics teacher, you are one of the best teachers in the world and you need to act like a teacher where
you can help student understand the world of math with being able to notice weak spots and able to make visualisations and simulate
what is happening to help better understanding.
"""

# ==========================================
# 3. MEMORY: The Context Window
# ==========================================
chat_history = [
    {"role": "user", "content": APP_PERSONA}
]

# ==========================================
# 4. THE MAIN APP LOOP
# ==========================================
print("\n--- App Started! Type 'exit' to quit. ---")

while True:
    # Get user input
    user_text = input("\nYou: ")

#VECTORIZE

    chat_history_list = []
    for i in range(len(chat_history)): # Corrected line: use len(chat_history)
      chat_history_list.append(chat_history[i]["content"])
    # The vectorizer needs to be initialized before use
    vectorizer = TfidfVectorizer()
    chat_vectors = vectorizer.fit_transform(chat_history_list) # Turn chat_history into numbers
    query_vector = vectorizer.transform([user_text]) # Turn question into numbers

    scores = cosine_similarity(query_vector, chat_vectors).flatten() # Corrected: use chat_vectors instead of chat_history_list


    best_index = np.argmax(scores)
    highest_score = scores[best_index]



    if user_text.lower() == 'exit':
        print("Closing application...")
        break

    # Add user's message to memory

    chat_history.append({"role": "user", "content": user_text})

    print("AI is thinking... :thinking:")
    # Send the whole history to the AI

    response = client.chat.completions.create(
            model="google/gemma-3-4b-it",
            messages=chat_history,
            temperature=0.7
        )

        # Get the AI's reply
    ai_reply = response.choices[0].message.content

        # Print it for the user
    print(f"\nAI: {ai_reply}")
        # Add the AI's reply to the memory
    chat_history.append({"role": "assistant", "content": ai_reply})

#_________PDF READER_________________


import PyPDF2
import requests
import io


pdf_url = input("Paste the PDF link here: ")

print("Downloading PDF...")

# Download the PDF
response = requests.get(pdf_url)

# Use BytesIO to treat the downloaded content as a file
pdf_file = io.BytesIO(response.content)
reader = PyPDF2.PdfReader(pdf_file)

# 2. This will hold all the text we find
massive_text_blob = ""

# 3. The "Page-by-Page" Loop

for page_number in range(len(reader.pages)): # Now iterate through reader.pages
    # Get the current page
    current_page = reader.pages[page_number]

    # Extract the text from that page
    text_from_page = current_page.extract_text()

    # Add the new text to our massive blob
    massive_text_blob += text_from_page + " "


UNCHUNKED = massive_text_blob

# Split into chunks of roughly 500 words
words = UNCHUNKED.split()

pdf_chunks = []

chunk_size = 500

for i in range(0, len(words), chunk_size):
    chunk = " ".join(words[i:i + chunk_size])
    pdf_chunks.append(chunk)

print(f"Created {len(pdf_chunks)} chunks.")


pdf_vectorizer = TfidfVectorizer()

pdf_vectors = pdf_vectorizer.fit_transform(pdf_chunks)

query_vector = pdf_vectorizer.transform([user_text])

scores = cosine_similarity(
    query_vector,
    pdf_vectors
).flatten()

top_indices = scores.argsort()[-3:][::-1]

relevant_text = "\n\n".join(
    pdf_chunks[i]
    for i in top_indices
)



messages = [
    {
        "role": "system",
        "content": APP_PERSONA
    },
    {
        "role": "system",
        "content": f"""
Relevant textbook information:

{relevant_text}

Use this information when answering.
If not use your own knowledge.
"""
    }
]

messages.extend(chat_history)

messages.append({
    "role": "user",
    "content": user_text
})

response = client.chat.completions.create(
    model="google/gemma-3-4b-it",
    messages=messages,
    temperature=0.7
)

use_pdf = input("Load a textbook PDF? (y/n): ")

if use_pdf.lower() == "y":
    pdf_url = input("PDF URL: ")

    response = requests.get(pdf_url)

    pdf_file = io.BytesIO(response.content)

    reader = PyPDF2.PdfReader(pdf_file)

    massive_text_blob = ""

    for page in reader.pages:
        text = page.extract_text()

        if text:
            massive_text_blob += text + " "

    print("Textbook loaded.")


#--------------------------------------
#--------------------------------------
#          VISUALS
#_____________________________________

if user_text.lower().startswith("graph "):

    equation_input = user_text[6:]

    x = sp.symbols('x')

    equation = sp.sympify(equation_input)

    graph_function = sp.lambdify(x, equation, 'numpy')

    x_vals = np.linspace(-10, 10, 400)
    y_vals = graph_function(x_vals)

    plt.figure(figsize=(8,5))
    plt.plot(x_vals, y_vals)

    plt.title(f"y = {equation}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(True)

    plt.show()

    continue


#CALCULATOR

import gradio as gr


# 1. The Brain (The Function)
def calculate(num1, operation, num2):
  if operation == "+":
    return num1 + num2
  elif operation == "-":
    return num1 - num2
  elif operation == "*":
    return num1 * num2
  elif operation == "/":
    if num2 == 0:
      return "Error: Division by zero"
    return num1 / num2
  else:
    return "Error: Invalid operation"

# 2. The Face (The Interface)
# We define the function, the inputs (two numbers), and the output(one number)
demo = gr.Interface(
  fn=calculate,
  inputs=[gr.Number(label="First Number"), gr.Textbox (label="Operation"), gr.Number(label="Second Number")],
  outputs=gr.Textbox(label="Result"),
  title="Calculator "
)
# 3. Launch!
demo.launch(share=True)


--- App Started! Type 'exit' to quit. ---

You: graph (sin)x
AI is thinking... :thinking:


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-4b-it is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'DeepInfra', 'is_byok': False}}, 'user_id': 'user_38xQMASEd7Kso32sMUuPwHH2KvC'}

CALCULATOR

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    a = gr.Number(label="First Number")
    b = gr.Number(label="Second Number")
    with gr.Row():
        add_btn = gr.Button("Add")
        sub_btn = gr.Button("Subtract")
        mul_btn = gr.Button("Multiply")
        div_btn = gr.Button("Divide")
    c = gr.Number(label="Result")

    def add(num1, num2):
        return num1 + num2
    add_btn.click(add, inputs=[a, b], outputs=c)

    def sub(data):
        return data[a] - data[b]
    sub_btn.click(sub, inputs={a, b}, outputs=c)

    def mul(num1, num2):
        return num1 * num2
    mul_btn.click(mul, inputs=[a, b], outputs=c)

    def div(data):
        return data[a] / data[b]
    div_btn.click(div, inputs={a, b}, outputs=c)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://97a44182fec94fb863.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ==========================================
# 1. THE DATA: Our "Knowledge Base"
# ==========================================
chunks = [
    "Kinetic Energy is the energy an object possesses",      # Chunk 0
    "Space exploration is very exciting.",          # Chunk 1
    "Money is the future",    # Chunk 2
    "I love eating delicious chocolate cake."       # Chunk 3
]

# ==========================================
# 2. THE USER QUERY
# ==========================================
user_query = "What does kinetic energy mean?"
print(f"User Query: {user_query}\n")
# ==========================================
# 3. BRAIN 1: KEYWORD SEARCH (BM25/TF-IDF)
# ==========================================
# We use TfidfVectorizer to find exact word matches
keyword_vectorizer = TfidfVectorizer()
keyword_matrix = keyword_vectorizer.fit_transform(chunks)
query_vector = keyword_vectorizer.transform([user_query])

# Get the scores (how well the words match)
keyword_scores = keyword_matrix.dot(query_vector.transpose()).toarray().flatten()

# ==========================================
# 4. BRAIN 2: SEMANTIC SEARCH (Simulated)
# ==========================================
# In a real app, this would come from an AI model (Embeddings).
# For this lesson, we will "fake" the scores to show the math works.
# We'll assume Chunk 0 and 2 are "semantically" related to the query.
# semantic_scores = np.array([0.95, 0.05, 0.85, 0.01])
# Calculate the cosine similarity between the user query vector and all chunk vectors.
similarity_scores = cosine_similarity(query_vector, keyword_matrix).flatten()
# Let's see the scores! It's a 2D array, so we access the first (and only) row.
print("Similarity Scores:", similarity_scores)

# ==========================================
# 5. THE HYBRID COMBINATION (The Math!)
# ==========================================
# We decide: 40% Keyword, 60% Semantic
weight_keyword = 0.4
weight_semantic = 0.6

hybrid_scores = []

# We loop through every chunk and calculate its total weighted score
for i in range(len(chunks)):
    total = (weight_keyword * keyword_scores[i]) + (weight_semantic * similarity_scores[i])
    hybrid_scores.append(total)

# ==========================================
# 6. THE RESULTS
# ==========================================
# Find the best index
best_index = np.argmax(hybrid_scores)

print("--- SEARCH RESULTS ---")
for i in range(len(chunks)):
    print(f"Chunk {i} | Score: {hybrid_scores[i]:.4f} | Text: {chunks[i]}")

print(f"\n🏆 WINNER: Chunk {best_index}")
print(f"Final Answer: {chunks[best_index]}")

User Query: What does kinetic energy mean?

Similarity Scores: [0.70597018 0.         0.         0.        ]
--- SEARCH RESULTS ---
Chunk 0 | Score: 0.7060 | Text: Kinetic Energy is the energy an object possesses
Chunk 1 | Score: 0.0000 | Text: Space exploration is very exciting.
Chunk 2 | Score: 0.0000 | Text: Money is the future
Chunk 3 | Score: 0.0000 | Text: I love eating delicious chocolate cake.

🏆 WINNER: Chunk 0
Final Answer: Kinetic Energy is the energy an object possesses


In [ ]:
!pip install PyPDF2

In [ ]:
import PyPDF2
import requests
import io

pdf_url = input("Paste the PDF link here: ")

print("Downloading PDF...")

# Download the PDF
response = requests.get(pdf_url)

# Check if download worked
if response.status_code != 200:
    print(f"Failed to download PDF")
    exit()

print("PDF downloaded successfully!")

# Use BytesIO to treat the downloaded content as a file
pdf_file = io.BytesIO(response.content)
reader = PyPDF2.PdfReader(pdf_file)

# 2. This will hold all the text we find
massive_text_blob = ""

# 3. The "Page-by-Page" Loop
print("Starting to scan the PDF...")

for page_number in range(len(reader.pages)): # Now iterate through reader.pages
    # Get the current page
    current_page = reader.pages[page_number]

    # Extract the text from that page
    text_from_page = current_page.extract_text()

    # Add the new text to our massive blob
    massive_text_blob += text_from_page + " "
    print({page_number})

print("Scanning complete!")


# Now, you would take 'massive_text_blob' and pass it to your
# Chunking code from the previous section!

UNCHUNKED = massive_text_blob







# 2. Settings for our chunks
all_chunks = UNCHUNKED.split(".") # An empty list to hold our pieces

# 3. Check our work
print(f"Original Text Length: {len(UNCHUNKED)}")
print(f"Number of Chunks Created: {len(all_chunks)}")
print("--- The Chunks ---")
for c in all_chunks:
    print(f"[{c}]")

# VISUALS

In [ ]:
pip install sympy matplotlib numpy

In [ ]:


import numpy as np
import matplotlib.pyplot as plt
import sympy as sp


x = sp.symbols('x')
equation_input = input("Enter in a math equation")

equation = sp.sympify(equation_input)
derivative = sp.diff(equation, x)
integral = sp.integrate(equation, x)

graph_function = sp.lambdify(x, equation, 'numpy')

x_vals = np.linspace(-10, 10, 400)
y_vals = graph_function(x_vals)


plt.plot(x_vals, y_vals)
plt.title("y = x²")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)

plt.show()